## Setup

In [11]:
from arcgis.gis import GIS
import arcpy
from arcgis.features import FeatureLayer
import pandas as pd
import json
import urllib.request
import urllib.parse
import os
import glob
import tempfile
import time
import requests

In [14]:
gis = GIS('pro')
print(gis)
aprx = arcpy.mp.ArcGISProject("CURRENT")
map = aprx.activeMap
arcpy.env.addOutputsToMap = False
arcpy.env.overwriteOutput = True
project_path = r"C:\Users\micha\OneDrive\Documents\ArcGIS\Projects\California Solar Suitability Sandbox"
working_gdb = os.path.join(project_path, r"California Solar Suitability Sandbox.gdb")

GIS @ https://www.arcgis.com/ version:[2026, 2]


In [ ]:
# Getting all the TIF files
download_list = pd.read_csv(os.path.join(project_path, r"DEMs\data_scrubbed.csv"))

output_folder = os.path.join(project_path, r"DEMs")
os.makedirs(output_folder, exist_ok=True)

for url in download_list['Link']:
    filename = os.path.join(output_folder, os.path.basename(url))
    urllib.request.urlretrieve(url, filename)
    print(f"Downloaded {filename}")

In [ ]:
# Combine files of interest into one raster dataset

tif_files = glob.glob(os.path.join(project_path, "DEMs", "*.tif"))

arcpy.management.MosaicToNewRaster(
    input_rasters=tif_files,
    output_location=working_gdb,
    raster_dataset_name_with_extension="dem_mosaic",
    pixel_type="32_BIT_FLOAT",
    number_of_bands=1
)

In [ ]:
# Let's reproject for meters
arcpy.management.ProjectRaster(
    in_raster=os.path.join(working_gdb, "dem_mosaic"),
    out_raster=os.path.join(working_gdb, "dem_mosaic_projected"),
    out_coor_system=arcpy.SpatialReference(3310)  # California Albers, a standard CA projected system in meters
)

In [ ]:
# Fill the DEM data just in case I need to do floodplain stuff later
filled_dem = arcpy.sa.Fill(os.path.join(working_gdb, "dem_mosaic_projected"))
filled_dem.save(os.path.join(working_gdb, "dem_clean"))
arcpy.management.Delete(os.path.join(working_gdb, "dem_mosaic_projected"))

In [9]:
# Creating a slope raster to find flat areas for solar
slope = arcpy.sa.Slope(
    in_raster=os.path.join(working_gdb, "dem_clean"),
    output_measurement="PERCENT_RISE"
)
slope.save(os.path.join(working_gdb, "dem_slope"))

In [ ]:
# Okay here I was checking the spatial reference on all my server-side exclusions to make sure everything is going to process together fine
# I ended up removing these from the map so I've commented this cell out
# print(arcpy.Describe("BaseExclusions_Solar_v2").spatialReference.name)
# print(arcpy.Describe("Technoeconomic_Exclusion_Solar").spatialReference.name)
# print(arcpy.Describe("ProtectedAreas_Solar").spatialReference.name)
# print(arcpy.Describe(os.path.join(working_gdb, "dem_slope")).spatialReference.name)

In [ ]:
# Here I'm clipping my exclusion polygons to my DEM data just to reduce geoprocessing time
layers_to_clip = {
    "BaseExclusions_Solar_v2": "Exclusions_combined",
    "Technoeconomic_Exclusion_Solar": "Exclusions_tech_econ",
    "ProtectedAreas_Solar": "Exclusions_protected"
}

arcpy.ddd.RasterDomain(
    in_raster=os.path.join(working_gdb, "dem_clean"),
    out_feature_class=os.path.join(working_gdb, "dem_extent"),
    out_geometry_type="POLYGON"
)

boundary = os.path.join(working_gdb, "dem_extent")

for source_name, output_name in layers_to_clip.items():
    arcpy.analysis.Clip(
        in_features=source_name,
        clip_features=boundary,
        out_feature_class=os.path.join(working_gdb, output_name)
    )
    print(f"Clipped {source_name} -> {output_name}")

In [ ]:
# Let's convert our combined exclusion polygon to a raster
arcpy.conversion.PolygonToRaster(
    in_features=os.path.join(working_gdb, "Exclusions_combined"),
    value_field="OBJECTID",
    out_rasterdataset=os.path.join(working_gdb, "Exclusions_raster"),
    cellsize=os.path.join(working_gdb, "dem_clean")
)

In [50]:
# Now we're going to create 5 categories for the slope
slope_reclass = arcpy.sa.Reclassify(
    in_raster=os.path.join(working_gdb, "dem_slope"),
    reclass_field="VALUE",
    remap=arcpy.sa.RemapRange([
        [0, 5, 5],      # 0-5% slope: best (5)
        [5, 10, 4],
        [10, 15, 3],
        [15, 25, 2],
        [25, 9999, 1]   # anything steeper: worst (1)
    ])
)
slope_reclass.save(os.path.join(working_gdb, "suitability_slope"))

In [5]:
# Let's reclassify the exclusion raster to 1 or 0, so we can just multiply by our slope score to get 1-5 (not an exclusion zone) or 0 (exclusion zone)
excl_reclass = arcpy.sa.Reclassify(
    in_raster=os.path.join(working_gdb, "Exclusions_raster"),
    reclass_field="VALUE",
    remap=arcpy.sa.RemapRange([[0, 999999, 1]]),  # anywhere with a real value = excluded = worst score
    missing_values="NODATA"  # NoData (not excluded) stays as-is, handled separately below
)

excl_suitability = arcpy.sa.Con(
    arcpy.sa.IsNull(os.path.join(working_gdb, "Exclusions_raster")), 1, 0
)
excl_suitability.save(os.path.join(working_gdb, "suitability_exclusions"))

In [ ]:
# I was having issues with extents so I copied a snapped dataset
arcpy.env.snapRaster = os.path.join(working_gdb, "suitability_slope")
arcpy.env.extent = os.path.join(working_gdb, "suitability_slope")

arcpy.management.CopyRaster(
    in_raster=os.path.join(working_gdb, "suitability_exclusions"),
    out_rasterdataset=os.path.join(working_gdb, "suitability_exclusions_snapped")
)

In [7]:
# Checking that it looks good
print(arcpy.Describe(os.path.join(working_gdb, "suitability_slope")).extent)
print(arcpy.Describe(os.path.join(working_gdb, "suitability_exclusions_snapped")).extent)

184598.031832124 -665933.586486625 473574.089883843 -433796.623178054 NaN NaN NaN NaN
184598.031832124 -665933.586486625 473574.089883843 -433796.623178054 NaN NaN NaN NaN


In [22]:
# Here I downloaded a bunch of csvs for irradiance data in my project area. This takes them all and pulls their location and avg GHI
results = []
for filepath in glob.glob(r"C:\Users\micha\OneDrive\Documents\ArcGIS\Projects\California Solar Suitability Sandbox\Irradiance\*.csv"):
    df = pd.read_csv(filepath, skiprows=2)  # skip NSRDB's 2-row metadata header
    filename = os.path.basename(filepath)          # "211031_33.93_-117.86_2025.csv"
    name, ext = os.path.splitext(filename)          # "211031_33.93_-117.86_2025", ".csv"
    point_id, lat, lon, year = name.split("_")
    results.append({
        "point_id": point_id,
        "lat": float(lat),
        "lon": float(lon),
        "mean_ghi": df["GHI"].mean()
    })

ghi_summary = pd.DataFrame(results)
ghi_summary.to_csv("ghi_by_point.csv", index=False)

In [23]:
# Now we take that GHI csv and create a raster

# Create a feature class
arcpy.management.XYTableToPoint(
    in_table=os.path.join(project_path,"ghi_by_point.csv"),
    out_feature_class=os.path.join(working_gdb, "ghi_points"),
    x_field="lon",
    y_field="lat",
    coordinate_system=arcpy.SpatialReference(4326)  # WGS84, since lat/lon
)
# Reproject for meters
arcpy.management.Project(
    in_dataset="ghi_points",
    out_dataset=os.path.join(working_gdb, "ghi_points_proj"),
    out_coor_system=arcpy.Describe("dem_clean").spatialReference
)
arcpy.management.Delete(os.path.join(working_gdb,"ghi_points"))

# Generate a raster
ghi_raster = arcpy.sa.Idw("ghi_points_proj", "mean_ghi", cell_size=arcpy.Describe("dem_clean").meanCellWidth)
ghi_raster.save(os.path.join(working_gdb, "ghi_surface"))
arcpy.management.Delete(os.path.join(working_gdb, "ghi_points_proj"))

<Result 'true'>

In [24]:
# Now we're going to create 5 categories for the irradiance
slope_reclass = arcpy.sa.Reclassify(
    in_raster=os.path.join(working_gdb, "ghi_surface"),
    reclass_field="VALUE",
    remap=arcpy.sa.RemapRange([
        [251, 257, 5],      # 0-5% slope: best (5)
        [249, 251, 4],
        [244, 249, 3],
        [231, 244, 2],
        [0,231, 1]   # anything lower: worst (1)
    ])
)
slope_reclass.save(os.path.join(working_gdb, "suitability_irradiance"))